In [2]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.1 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import faiss
import pickle

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
movies = pd.read_csv("processed_movies.csv")

embeddings = np.load("movie_embeddings.npy")

index = faiss.read_index("faiss.index")

svd = pickle.load(open("svd_model.pkl","rb"))

In [5]:
movie_name = "Interstellar"

movie_index = movies[
    movies["title"] == movie_name
].index[0]

In [6]:
query = embeddings[movie_index].reshape(1,-1)

scores, indices = index.search(query,50)

In [7]:
candidate_movies = movies.iloc[indices[0]].copy()

candidate_movies.head()

,id,title,tags,vote_average,vote_count,popularity,release_date,original_language
22805,157336,Interstellar,interstellar chronicl the adventur of a group ...,8.1,11187.0,32.213481,2014-11-05,en
41799,274870,Passengers,a spacecraft travel to a distant coloni planet...,6.7,4134.0,20.303632,2016-12-21,en
13192,10200,The Day the Earth Stood Still,a repres of an alien race that went through dr...,5.2,1057.0,9.265322,2008-12-10,en
1451,18,The Fifth Element,"in 2257, a taxi driver is unintent given the t...",7.3,3962.0,24.305260,1997-05-07,en
4494,4296,Millennium,an investig seek the caus of an airlin disast ...,5.0,45.0,2.742085,1989-08-25,en


In [9]:
candidate_movies["popularity_score"] = (

candidate_movies["popularity"]

/

candidate_movies["popularity"].max()
)

In [10]:
candidate_movies["vote_score"] = (

candidate_movies["vote_average"]

/

10

)

In [11]:
candidate_movies["vote_score"] = (

candidate_movies["vote_average"]

/

10

)

In [12]:
user_id = 1

In [13]:
collab_scores = []

for movie_id in candidate_movies["id"]:

    try:

        prediction = svd.predict(

            user_id,

            movie_id

        )

        collab_scores.append(

            prediction.est

        )

    except:

        collab_scores.append(

            0

        )

candidate_movies["collab_score"] = collab_scores

In [14]:
candidate_movies["collab_score"] = (

candidate_movies["collab_score"]

/

5

)

In [15]:
semantic_scores = scores[0]

semantic_scores = (

semantic_scores

-

semantic_scores.min()

)

/

(

semantic_scores.max()

-

semantic_scores.min()

)

candidate_movies["semantic_score"] = semantic_scores

In [16]:
candidate_movies["final_score"] = (

0.50

*

candidate_movies["semantic_score"]

+

0.30

*

candidate_movies["collab_score"]

+

0.20

*

candidate_movies["vote_score"]

)

In [17]:
candidate_movies = candidate_movies.sort_values(

    by="final_score",

    ascending=False

)

In [18]:
candidate_movies[

[
"title",
"final_score",
"vote_average",
"release_date"

]

].head(10)

,title,final_score,vote_average,release_date
22805,Interstellar,0.602681,8.1,2014-11-05
41799,Passengers,0.406659,6.7,2016-12-21
16104,Woman in the Moon,0.395410,6.7,1929-10-14
1451,The Fifth Element,0.392583,7.3,1997-05-07
25092,La Belle verte,0.382984,7.5,1996-09-18
6970,Royal Space Force - The Wings Of Honneamise,0.376934,6.7,1987-03-14
22117,Her,0.371157,7.9,2013-12-18
40903,Arrival,0.369055,7.2,2016-11-10
10253,Serenity,0.367942,7.4,2005-08-25
20878,Star Trek Into Darkness,0.367325,7.4,2013-05-05


In [19]:
candidate_movies.to_csv(

    "hybrid_recommendations.csv",

    index=False

)